In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns



# Load data
data_frame= pd.read_csv('/mnt/disk15tb/paula/Main_DA_Projects/data_analysis_output/Primary Neurons/CDKL5_T2_Jan29/Network_outputs/Compiled_Networks.csv')  # Ensure to change the path as needed
data_frame.columns

In [ ]:
data_frame.columns

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy.ndimage import gaussian_filter1d
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages

# Define the output folder
output_folder = "/mnt/disk15tb/paula/Main_DA_Projects/data_analysis_output/Primary Neurons/CDKL5_T2_Jan29/Network_outputs/plots/"
os.makedirs(output_folder, exist_ok=True)

# Initialize the PDF file
pdf_filename = os.path.join(output_folder, "all_plots.pdf")
pdf_pages = PdfPages(pdf_filename)

# Copy the original DataFrame
temp_df = data_frame.copy()

# Columns to process
columns_to_process = ['Burst_Peak_List', 'Abs_Burst_Peak_List', 'Burst_Times_List', 'IBI_List', 'SpikesPerBurst_List']

for column in columns_to_process:
    # Step 1: Convert column strings to lists of floats
    temp_df[column] = temp_df[column].apply(lambda x: [float(i) for i in x.split(",")])

    # Step 2: Explode the lists into individual rows
    exploded_df = temp_df.explode(column)

    # Rename for clarity
    exploded_df.rename(columns={column: "Value"}, inplace=True)

    # Convert to numeric (in case there are any remaining non-numeric values)
    exploded_df["Value"] = pd.to_numeric(exploded_df["Value"], errors="coerce")

    # Apply log transformation for the second set of plots
    exploded_df["LogValue"] = np.log10(exploded_df["Value"] + 1)

    # Step 3: Group data for binning and calculate probabilities (both original and log-transformed)
    group_counts, log_group_counts = [], []
    bins = np.histogram_bin_edges(exploded_df["Value"], bins='auto')
    log_bins = np.histogram_bin_edges(exploded_df["LogValue"], bins='auto')

    # Adjust grouping to include ChipID, NeuronType, and Well
    for (div, chip_id, neuron_type, well), group in exploded_df.groupby(["DIV", "Chip_ID", "NeuronType", "Well"]):
        # Original value counts and probabilities
        counts, _ = np.histogram(group["Value"], bins=bins)
        probabilities = counts / counts.sum()
        smoothed_probabilities = gaussian_filter1d(probabilities, sigma=1)

        # Log-transformed value counts and probabilities
        log_counts, _ = np.histogram(group["LogValue"], bins=log_bins)
        log_probabilities = log_counts / log_counts.sum()
        log_smoothed_probabilities = gaussian_filter1d(log_probabilities, sigma=1)

        # Append to respective group lists
        for bin_left, prob in zip(bins[:-1], smoothed_probabilities):
            group_counts.append({
                "DIV": div,
                "ChipID": chip_id,
                "NeuronType": neuron_type,
                "Well": well,
                "Bin_Start": bin_left,
                "Probability": prob
            })

        for bin_left, prob in zip(log_bins[:-1], log_smoothed_probabilities):
            log_group_counts.append({
                "DIV": div,
                "ChipID": chip_id,
                "NeuronType": neuron_type,
                "Well": well,
                "Bin_Start": bin_left,
                "Probability": prob
            })

    # Convert to DataFrames
    smoothed_df = pd.DataFrame(group_counts)
    log_smoothed_df = pd.DataFrame(log_group_counts)

    # Step 4: Generate plots
    for df_to_plot, transformation, file_suffix in zip(
            [smoothed_df, log_smoothed_df],
            ["Original", "Log-Transformed"],
            ["original", "log"]):
        
        wells = df_to_plot["Well"].unique()
        num_rows, num_cols = (len(wells) + 1) // 2, 2
        fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 6 * num_rows), sharex=True, sharey=True)
        axes = axes.flatten()

        for i, well in enumerate(wells):
            ax = axes[i]

            subset = df_to_plot[df_to_plot["Well"] == well]

            sns.lineplot(
                data=subset,
                x="Bin_Start",
                y="Probability",
                hue="DIV",
                marker="o",
                ax=ax
            )

            ax.set_title(f"Chip {chip_id} - {neuron_type} - Well {well} - {transformation} Distribution", fontsize=10)
            ax.set_ylabel("Probability")
            ax.set_xlabel("Value Range" if transformation == "Original" else "Log(Value Range)")
            ax.legend(title="DIV", fontsize=8, loc="upper right")
            ax.grid()

        for i in range(len(wells), len(axes)):
            fig.delaxes(axes[i])

        # Save individual plot
        individual_plot_filename = os.path.join(output_folder, f"{column}_{chip_id}_{neuron_type}_{file_suffix}_distribution.png")
        plt.tight_layout()
        plt.suptitle(f"{column} - {chip_id} - {neuron_type} - {transformation} Across Wells", fontsize=16, y=1.02)
        plt.savefig(individual_plot_filename, dpi=300)

        # Add the plot to the PDF
        pdf_pages.savefig(fig)
        plt.show()
        plt.close()

# Close the PDF file
pdf_pages.close()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.ndimage import gaussian_filter1d
from matplotlib.backends.backend_pdf import PdfPages

# Define the output folder
output_folder = "/mnt/disk15tb/paula/Main_DA_Projects/data_analysis_output/Primary Neurons/CDKL5_T2_Jan29/Network_outputs/plots2/"
os.makedirs(output_folder, exist_ok=True)

# Create a single PDF for all plots
pdf_filename = os.path.join(output_folder, "all_plots.pdf")
pdf_pages = PdfPages(pdf_filename)

# Columns to process
columns_to_process = ['Burst_Peak_List', 'Abs_Burst_Peak_List', 'Burst_Times_List', 'IBI_List', 'SpikesPerBurst_List']

for column in columns_to_process:
    # Step 1: Convert column strings to lists of floats without altering the original dataframe
    temp_df = data_frame.copy()
    temp_df[column] = temp_df[column].apply(lambda x: [float(i) for i in x.split(",")])

    # Step 2: Explode the lists into individual rows
    exploded_df = temp_df.explode(column)

    # Rename for clarity
    exploded_df.rename(columns={column: "Value"}, inplace=True)

    # Convert to numeric (in case there are any remaining non-numeric values)
    exploded_df["Value"] = pd.to_numeric(exploded_df["Value"], errors="coerce")
    # Add a log-transformed column
    exploded_df["LogValue"] = np.log10(exploded_df["Value"] + 1)

    # Step 3: Define bins automatically for histogram computation
    bins = np.histogram_bin_edges(exploded_df["Value"], bins='auto')  # Automatic bin selection
    log_bins = np.histogram_bin_edges(exploded_df["LogValue"], bins='auto')
    
    # Step 4: Compute histogram counts and probabilities
    group_counts = []
    log_group_counts = []

    # Group by DIV, Chip_ID, NeuronType, and Well
    for (div, chip_id, neuron_type, well), group in exploded_df.groupby(["DIV", "Chip_ID", "NeuronType", "Well"]):
        counts, _ = np.histogram(group["Value"], bins=bins)
        probabilities = counts / counts.sum()
        smoothed_probabilities = gaussian_filter1d(probabilities, sigma=1)

        log_counts, _ = np.histogram(group["LogValue"], bins=log_bins)
        log_probabilities = log_counts / log_counts.sum()
        log_smoothed_probabilities = gaussian_filter1d(log_probabilities, sigma=1)

        # Original values
        for bin_left, prob in zip(bins[:-1], smoothed_probabilities):
            group_counts.append({
                "DIV": div,
                "Chip_ID": chip_id,
                "NeuronType": neuron_type,
                "Well": well,
                "Bin_Start": bin_left,
                "Probability": prob
            })

        # Log-transformed values
        for bin_left, prob in zip(log_bins[:-1], log_smoothed_probabilities):
            log_group_counts.append({
                "DIV": div,
                "Chip_ID": chip_id,
                "NeuronType": neuron_type,
                "Well": well,
                "Bin_Start": bin_left,
                "Probability": prob
            })

    # Convert to DataFrames
    smoothed_df = pd.DataFrame(group_counts)
    log_smoothed_df = pd.DataFrame(log_group_counts)

    # Step 5: Create a custom palette
    wells = smoothed_df["Well"].unique()
    palette = {}
    for well in wells:
        if well in [1, 2, 3]:  # Wells 1–3: Blue hues
            palette[well] = sns.color_palette("Blues", 3)[well - 1]
        elif well in [4, 5, 6]:  # Wells 4–6: Red hues
            palette[well] = sns.color_palette("Reds", 3)[well - 4]

    # Step 6: Create subplots for each DIV
    divs = smoothed_df["DIV"].unique()
    num_rows, num_cols = (len(divs) + 1) // 2, 2  # Adjust rows/columns based on DIV count
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 6 * num_rows), sharex=True, sharey=True)

    axes = axes.flatten()  # Flatten axes for easier indexing

    for i, div in enumerate(divs):
        ax = axes[i]

        subset = smoothed_df[smoothed_df["DIV"] == div]
        
        # Plot original values with the custom palette
        for well, well_data in subset.groupby("Well"):
            sns.lineplot(
                data=well_data,
                x="Bin_Start",
                y="Probability",
                color=palette[well],  # Assign color based on well
                label=f"Well{well}",
                marker="o",
                ax=ax
            )

        # Customize plot appearance
        ax.set_title(f"DIV {div} - Smoothed Distribution (Original)", fontsize=10)
        ax.set_ylabel("Probability")
        ax.set_xlabel("Value Range")
        ax.legend(title="Well", fontsize=8, loc="upper right")
        ax.grid()

    # Remove unused subplots if DIVs < num_rows * num_cols
    for i in range(len(divs), len(axes)):
        fig.delaxes(axes[i])

    # Save the individual plot
    individual_plot_filename = os.path.join(output_folder, f"{column}_original_distribution.png")
    plt.tight_layout()
    plt.suptitle(f"Smoothed Probability Distribution of {column} (Original) Across DIVs", fontsize=16, y=1.02)
    plt.savefig(individual_plot_filename, dpi=300)

    # Add to the PDF
    pdf_pages.savefig(fig)
    plt.close()

    # Repeat the same process for log-transformed data
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 6 * num_rows), sharex=True, sharey=True)
    axes = axes.flatten()

    for i, div in enumerate(divs):
        ax = axes[i]

        subset = log_smoothed_df[log_smoothed_df["DIV"] == div]
        
        # Plot log-transformed values with the custom palette
        for well, well_data in subset.groupby("Well"):
            sns.lineplot(
                data=well_data,
                x="Bin_Start",
                y="Probability",
                color=palette[well],
                label=f"Well {well}",
                marker="o",
                ax=ax
            )

        # Customize plot appearance
        ax.set_title(f"DIV {div} - Smoothed Distribution (Log-Transformed)", fontsize=10)
        ax.set_ylabel("Probability")
        ax.set_xlabel("Log(Value Range)")
        ax.legend(title="Well", fontsize=8, loc="upper right")
        ax.grid()

    # Remove unused subplots if DIVs < num_rows * num_cols
    for i in range(len(divs), len(axes)):
        fig.delaxes(axes[i])

    # Save the individual plot
    individual_plot_filename = os.path.join(output_folder, f"{column}_log_distribution.png")
    plt.tight_layout()
    plt.suptitle(f"Smoothed Probability Distribution of {column} (Log-Transformed) Across DIVs", fontsize=16, y=1.02)
    plt.savefig(individual_plot_filename, dpi=300)

    # Add to the PDF
    pdf_pages.savefig(fig)
    plt.close()

# Close the PDF
pdf_pages.close()

In [ ]:
from scipy.ndimage import gaussian_filter1d
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Columns to process
columns_to_process = ['Burst_Peak_List', 'Abs_Burst_Peak_List', 'Burst_Times_List', 'IBI_List', 'SpikesPerBurst_List']

for column in columns_to_process:
    # Step 1: Convert column strings to lists of floats without altering the original dataframe
    temp_df = df.copy()
    temp_df[column] = temp_df[column].apply(lambda x: [float(i) for i in x.split(",")])

    # Step 2: Explode the lists into individual rows
    exploded_df = temp_df.explode(column)

    # Rename for clarity
    exploded_df.rename(columns={column: "Value"}, inplace=True)

    # Step 3: Define bins automatically for histogram computation
    bins = np.histogram_bin_edges(exploded_df["Value"], bins='auto')  # Automatic bin selection
    bin_labels = [f"{bins[i]:.2f} - {bins[i+1]:.2f}" for i in range(len(bins) - 1)]

    # Step 4: Compute histogram counts and probabilities
    group_counts = []

    for (div, well), group in exploded_df.groupby(["DIV", "Well"]):
        counts, _ = np.histogram(group["Value"], bins=bins)
        probabilities = counts / counts.sum()  # Normalize to probabilities
        # Apply Gaussian smoothing to probabilities
        smoothed_probabilities = gaussian_filter1d(probabilities, sigma=1)

        for bin_left, prob in zip(bins[:-1], smoothed_probabilities):
            group_counts.append({
                "DIV": div,
                "Well": well,
                "Bin_Start": bin_left,
                "Probability": prob
            })

    # Convert to DataFrame
    smoothed_df = pd.DataFrame(group_counts)

    # Step 5: Create a custom palette
    wells = smoothed_df["Well"].unique()
    palette = {}
    for well in wells:
        if well in [1, 2, 3]:  # Wells 1–3: Blue hues
            palette[well] = sns.color_palette("Blues", 3)[well - 1]
        elif well in [4, 5, 6]:  # Wells 4–6: Red hues
            palette[well] = sns.color_palette("Reds", 3)[well - 4]

    # Step 6: Create a subplot for each DIV
    divs = smoothed_df["DIV"].unique()
    num_rows, num_cols = (len(divs) + 1) // 2, 2  # Adjust rows/columns based on DIV count
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 6 * num_rows), sharex=True, sharey=True)

    axes = axes.flatten()  # Flatten axes for easier indexing

    for i, div in enumerate(divs):
        ax = axes[i]

        subset = smoothed_df[smoothed_df["DIV"] == div]
        
        # Plot with the custom palette
        for well, well_data in subset.groupby("Well"):
            sns.lineplot(
                data=well_data,
                x="Bin_Start",
                y="Probability",
                color=palette[well],  # Assign color based on well
                label=f"Well {well}",
                marker="o",
                ax=ax
            )

        # Customize plot appearance
        ax.set_title(f"DIV {div} - Smoothed Distribution", fontsize=10)
        ax.set_ylabel("Probability")
        ax.set_xlabel("Value Range")
        ax.legend(title="Well", fontsize=8, loc="upper right")
        ax.grid()

    # Remove unused subplots if DIVs < num_rows * num_cols
    for i in range(len(divs), len(axes)):
        fig.delaxes(axes[i])

    # Adjust layout and show the plot
    plt.tight_layout()
    plt.suptitle(f"Smoothed Probability Distribution of {column} Across DIVs", fontsize=16, y=1.02)
    plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages

# Define output folder (adjust to your actual path)
output_folder = "/mnt/disk15tb/paula/Main_DA_Projects/data_analysis_output/Primary Neurons/CDKL5_T2_Jan29/Network_outputs/plots_boxplot/"
os.makedirs(output_folder, exist_ok=True)

# Define a PDF file to collect all plots
pdf_filename = os.path.join(output_folder, "boxplots_scatter.pdf")
pdf_pages = PdfPages(pdf_filename)

# Columns to process
columns_to_process = ['Burst_Peak_List', 'Abs_Burst_Peak_List', 'Burst_Times_List', 'IBI_List', 'SpikesPerBurst_List']

for column in columns_to_process:
    # Step 1: Convert column strings to lists of floats without altering the original dataframe
    temp_df = data_frame.copy()
    temp_df[column] = temp_df[column].apply(lambda x: [float(i) for i in x.split(",")])

    # Step 2: Explode the lists into individual rows
    exploded_df = temp_df.explode(column)

    # Rename for clarity
    exploded_df.rename(columns={column: "Value"}, inplace=True)

    # Step 3: Assign groups (WT for wells 1–3, HET for wells 4–6)
    exploded_df["Group"] = exploded_df["Well"].apply(lambda x: "WT" if x in [1, 2, 3] else "HET")

    # Step 4: Create box plots with overlayed scatter points
    divs = exploded_df["DIV"].unique()
    num_rows, num_cols = (len(divs) + 1) // 2, 2  # Adjust rows/columns based on DIV count
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 6 * num_rows), sharex=True, sharey=True)
    axes = axes.flatten()

    for i, div in enumerate(divs):
        ax = axes[i]

        # Subset data for current DIV
        div_data = exploded_df[exploded_df["DIV"] == div]

        # Plot box plot
        sns.boxplot(
            data=div_data,
            x="Group",
            y="Value",
            ax=ax,
            palette={"WT": sns.color_palette("Blues")[2], "HET": sns.color_palette("Reds")[2]},
            width=0.6,
            showcaps=True,
            showmeans=False,
            boxprops=dict(alpha=0.7)  # Add transparency for the box
        )

        # Overlay scatter points (individual wells)
        sns.stripplot(
            data=div_data,
            x="Group",
            y="Value",
            hue="Well",
            palette={
                1: sns.color_palette("Blues", 3)[0],
                2: sns.color_palette("Blues", 3)[1],
                3: sns.color_palette("Blues", 3)[2],
                4: sns.color_palette("Reds", 3)[0],
                5: sns.color_palette("Reds", 3)[1],
                6: sns.color_palette("Reds", 3)[2]
            },
            dodge=False,  # Keep scatter points aligned with the box
            size=5,
            alpha=0.8,
            jitter=True,  # Slight jitter for better visibility
            ax=ax
        )

        # Customize plot appearance
        ax.set_title(f"DIV {div} - Box Plot with Overlayed Scatter", fontsize=10)
        ax.set_ylabel("Value")
        ax.set_xlabel("")
        ax.grid(axis="y")
        ax.legend(title="Well", fontsize=8, loc="upper right", frameon=False)

    # Remove unused subplots if DIVs < num_rows * num_cols
    for i in range(len(divs), len(axes)):
        fig.delaxes(axes[i])

    # Adjust layout
    plt.tight_layout()
    plt.suptitle(f"Box Plot with Scatter Overlay of {column} Across DIVs (WT vs HET)", fontsize=16, y=1.02)

    # Save the figure to the PDF
    pdf_pages.savefig(fig)
    plt.close()

# Close the PDF file after all plots are generated
pdf_pages.close()

In [ ]:
from itertools import combinations
import matplotlib.pyplot as plt
import os
from matplotlib.backends.backend_pdf import PdfPages

# Columns to process
columns_to_process = ['Burst_Peak_List', 'Burst_Times_List', 'IBI_List', 'SpikesPerBurst_List','Abs_Burst_Peak_List']

# Step 1: Convert the specified columns into individual lists of floats
temp_df = data_frame.copy()
for column in columns_to_process:
    temp_df[column] = temp_df[column].apply(lambda x: [float(i) for i in x.split(",")])

# Truncate lists to match the shortest length for each row
def truncate_lists(row, columns):
    min_length = min(len(row[col]) for col in columns)
    for col in columns:
        row[col] = row[col][:min_length]
    return row

temp_df = temp_df.apply(truncate_lists, columns=columns_to_process, axis=1)

# Generate all column combinations
column_combinations = list(combinations(columns_to_process, 2))

# Define output folder for saving PNGs
output_folder = "/mnt/disk15tb/paula/Main_DA_Projects/data_analysis_output/Primary Neurons/CDKL5_T2_Jan29/Network_outputs/plots_pairplots/"
os.makedirs(output_folder, exist_ok=True)

# Create a single PDF for all scatter plots
pdf_filename = os.path.join(output_folder, "pair_plots.pdf")
pdf_pages = PdfPages(pdf_filename)

# Generate scatter plots by DIV, coloring each Well differently
divs = temp_df['DIV'].unique()

for div in divs:
    subset = temp_df[temp_df['DIV'] == div]
    
    # Prepare a subplot grid for each DIV
    fig, axes = plt.subplots(len(column_combinations), 1, figsize=(8, 6 * len(column_combinations)))
    fig.suptitle(f"Scatter Plots for DIV {div}", fontsize=16, y=0.92)
    
    for (col_x, col_y), ax in zip(column_combinations, axes):
        ax.set_title(f"{col_x} vs {col_y}")
        ax.set_xlabel(col_x)
        ax.set_ylabel(col_y)
        
        for well, well_data in subset.groupby('Well'):
            x_values = well_data[col_x].explode()
            y_values = well_data[col_y].explode()
            
            # Blue hues for Wells 1-3, red hues for Wells 4-6
            if well <= 3:
                color = plt.cm.Blues(0.3 + (well - 1) / 3)
            else:
                color = plt.cm.Reds(0.3+(well - 4) / 3)
            
            ax.scatter(x_values, y_values, label=f"Well {well}", color=color, alpha=0.8)
        
        ax.legend(loc='upper left', fontsize=8)
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    
    # Save individual plot as PNG
    png_filename = os.path.join(output_folder, f"scatter_div_{div}.png")
    plt.savefig(png_filename, dpi=300)
    
    # Save plot to PDF
    pdf_pages.savefig(fig)
    plt.show()
    plt.close()

# Close the PDF file after all plots are generated
pdf_pages.close()

In [ ]:
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# Columns to process
columns_to_process = ['Burst_Peak_List', 'Burst_Times_List', 'IBI_List', 'SpikesPerBurst_List','Abs_Burst_Peak_List']

# Step 1: Convert column strings to lists of floats and apply log transformation
temp_df = data_frame.copy()
for column in columns_to_process:
    temp_df[column] = temp_df[column].apply(lambda x: np.log10(np.array([float(i) for i in x.split(",")]) + 1))

# Truncate lists to match the shortest length for each row
def truncate_lists(row, columns):
    min_length = min(len(row[col]) for col in columns)
    for col in columns:
        row[col] = row[col][:min_length]
    return row

temp_df = temp_df.apply(truncate_lists, columns=columns_to_process, axis=1)

# Generate all column combinations
column_combinations = list(combinations(columns_to_process, 2))

# Define output folder for saving PNGs
output_folder = "/mnt/disk15tb/paula/Main_DA_Projects/data_analysis_output/Primary Neurons/CDKL5_T2_Jan29/Network_outputs/plots_pairplots/"
os.makedirs(output_folder, exist_ok=True)

# Create a single PDF for all scatter plots
pdf_filename = os.path.join(output_folder, "log_transformed_scatter_plots.pdf")
pdf_pages = PdfPages(pdf_filename)

# Generate scatter plots by DIV, coloring each Well differently
divs = temp_df['DIV'].unique()

for div in divs:
    subset = temp_df[temp_df['DIV'] == div]
    
    # Prepare a subplot grid for each DIV
    fig, axes = plt.subplots(len(column_combinations), 1, figsize=(8, 6 * len(column_combinations)))
    fig.suptitle(f"Log-Transformed Scatter Plots for DIV {div}", fontsize=16, y=0.92)
    
    for (col_x, col_y), ax in zip(column_combinations, axes):
        ax.set_title(f"{col_x} vs {col_y}")
        ax.set_xlabel(f"log({col_x})")
        ax.set_ylabel(f"log({col_y})")
        
        for well, well_data in subset.groupby('Well'):
            x_values = well_data[col_x].explode()
            y_values = well_data[col_y].explode()
            
            # Blue hues for Wells 1-3, red hues for Wells 4-6
            if well <= 3:
                color = plt.cm.Blues(0.3+(well - 1) / 3)
            else:
                color = plt.cm.Reds(0.3+(well - 4) / 3)
            
            ax.scatter(x_values, y_values, label=f"Well {well}", color=color, alpha=0.8)
        
        ax.legend(loc='upper left', fontsize=8)
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    
    # Save individual plot as PNG
    png_filename = os.path.join(output_folder, f"log_scatter_pairwise_div_{div}.png")
    plt.savefig(png_filename, dpi=300)
    
    # Save plot to PDF
    pdf_pages.savefig(fig)
    plt.show()
    plt.close()

# Close the PDF file after all plots are generated
pdf_pages.close()

In [ ]:
# Columns to process
columns_to_process = ['Burst_Peak_List', 'Abs_Burst_Peak_List', 'Burst_Times_List', 'IBI_List', 'SpikesPerBurst_List']

# Step 1: Convert the specified columns into individual lists of floats
temp_df = df.copy()
for column in columns_to_process:
    temp_df[column] = temp_df[column].apply(lambda x: [float(i) for i in x.split(",")])
    print(len(temp_df[column]))



# Step 2: Explode all specified columns into individual rows
exploded_dataframes = []
for column in columns_to_process:
    exploded_df = temp_df.explode(column)
    exploded_df.rename(columns={column: "Value"}, inplace=True)
    exploded_df["Feature"] = column  # Add a column to indicate the feature name
    exploded_dataframes.append(exploded_df[["DIV", "Well", "Feature", "Value"]])

# Combine all the exploded DataFrames
combined_df = pd.concat(exploded_dataframes, axis=0)

# Get unique DIV values
unique_divs = combined_df["DIV"].unique()

# Get all unique features
unique_features = combined_df["Feature"].unique()
from itertools import combinations
# Generate all possible pairs of features
feature_pairs = list(combinations(unique_features, 2))
# Loop through each DIV
for div_value in unique_divs:
    # Filter data for this DIV
    div_df = combined_df[combined_df["DIV"] == div_value]
    
    # Pivot the table so that features become columns
    pivoted = div_df.pivot(index="Well", columns="Feature", values="Value")
    
    # Loop through each pair of features
    for feature_x, feature_y in feature_pairs:
        # Check if both features exist in the pivoted data
        if feature_x in pivoted.columns and feature_y in pivoted.columns:
            # Drop rows where the selected features aren't both present
            pivoted_clean = pivoted.dropna(subset=[feature_x, feature_y])
            
            # Plotting
            plt.figure(figsize=(8, 6))
            
            # Plot each well in a different color
            for well in pivoted_clean.index:
                plt.scatter(
                    pivoted_clean.loc[well, feature_x],
                    pivoted_clean.loc[well, feature_y],
                    label=f"Well {well}",
                    alpha=0.7
                )
            
            plt.xlabel(feature_x)
            plt.ylabel(feature_y)
            plt.title(f"DIV {div_value} - {feature_x} vs {feature_y}")
            plt.legend(title="Well")
            plt.grid(True)
            plt.show()